In [ ]:
# ==========================================
# 12 - CLASS WEIGHT EXPERIMENT
# KHỞI TẠO MÔI TRƯỜNG
# ==========================================

import sys
import json
from pathlib import Path

ROOT_DIR = Path.cwd().resolve().parents[2]
sys.path.insert(0, str(ROOT_DIR))

import src.imports as lib

from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    hamming_loss,
    precision_score,
    recall_score
)

RANDOM_SEED = 42
N_SPLITS = 5
MAX_WEIGHT = 20.0

lib.np.random.seed(RANDOM_SEED)

id_col = "ma_hoc_sinh"

choice_cols = [
    "Vật lí", "Hóa học", "Sinh học", "Địa lí",
    "Giáo dục kinh tế và pháp luật (GDKT&PL)",
    "Tin học", "Công nghệ", "Âm nhạc", "Mĩ thuật"
]

split_dir = ROOT_DIR / "data/data-splits"
fold_dir = split_dir / "cv_folds_v1"

output_dir = (
    ROOT_DIR
    / "experiments/v2_optimization/01_class_weight/results"
)

table_dir = output_dir / "tables"
config_dir = output_dir / "configs"

for directory in [table_dir, config_dir]:
    directory.mkdir(parents=True, exist_ok=True)

df_dev = lib.pd.read_csv(
    split_dir / "development.csv",
    encoding="utf-8-sig"
)

score_cols = [
    col for col in df_dev.columns
    if col.startswith(("L6_", "L7_", "L8_", "L9_"))
    and col.endswith(("_hki", "_hkii", "_ca_nam"))
]

X = df_dev[score_cols].copy()
y = df_dev[choice_cols].copy()

assert X.shape == (9000, 96)
assert y.shape == (9000, 9)
assert df_dev[id_col].is_unique
assert X.notna().all().all()
assert y.isin([0, 1]).all().all()
assert y.sum(axis=1).eq(4).all()

# Đọc 5 fold cố định
fold_data = []

for fold in range(1, N_SPLITS + 1):

    train_ids = lib.pd.read_csv(
        fold_dir / f"fold_{fold}_train_ids.csv"
    )[id_col]

    val_ids = lib.pd.read_csv(
        fold_dir / f"fold_{fold}_validation_ids.csv"
    )[id_col]

    assert set(train_ids).isdisjoint(set(val_ids))
    assert (
        set(train_ids) | set(val_ids)
        == set(df_dev[id_col])
    )

    train_idx = lib.np.flatnonzero(
        df_dev[id_col].isin(train_ids).to_numpy()
    )

    val_idx = lib.np.flatnonzero(
        df_dev[id_col].isin(val_ids).to_numpy()
    )

    fold_data.append({
        "fold": fold,
        "train_idx": train_idx,
        "val_idx": val_idx
    })

print("Development:", df_dev.shape)
print("X:", X.shape)
print("y:", y.shape)
print("Số fold:", len(fold_data))
print("Random Seed:", RANDOM_SEED)
print("PASS: Đã tải dữ liệu và 5 fold cố định.")

In [ ]:
# ==========================================
# PHÂN TÍCH CLASS IMBALANCE
# ==========================================

imbalance_records = []

for fold_info in fold_data:

    fold = fold_info["fold"]
    train_idx = fold_info["train_idx"]

    y_train = y.iloc[train_idx]

    for label in choice_cols:

        n_positive = int(y_train[label].sum())
        n_negative = int(len(y_train) - n_positive)

        ratio = (
            n_negative / n_positive
            if n_positive > 0
            else float("inf")
        )

        imbalance_records.append({
            "Fold": fold,
            "Nhãn": label,
            "Train_Count": len(y_train),
            "Positive": n_positive,
            "Negative": n_negative,
            "Positive_Rate_%": round(
                n_positive / len(y_train) * 100, 4
            ),
            "Negative_to_Positive": ratio
        })

imbalance_df = lib.pd.DataFrame(imbalance_records)

print("=" * 60)
print("CLASS IMBALANCE - TRAINING FOLDS")
print("=" * 60)

display(imbalance_df)

print("\nTRUNG BÌNH QUA 5 FOLD")

imbalance_summary = (
    imbalance_df.groupby("Nhãn", as_index=False)
    .agg({
        "Positive": "mean",
        "Negative": "mean",
        "Positive_Rate_%": "mean",
        "Negative_to_Positive": "mean"
    })
)

display(imbalance_summary.round(4))

imbalance_df.to_csv(
    table_dir / "class_imbalance_5fold.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
# ==========================================
# CLASS WEIGHT STRATEGIES
# ==========================================

WEIGHT_STRATEGIES = [
    "balanced",
    "sqrt_balanced"
]

def calculate_class_weight(y_train_label, strategy):

    n_positive = int(y_train_label.sum())
    n_negative = int(len(y_train_label) - n_positive)

    if n_positive == 0 or n_negative == 0:
        raise ValueError(
            "Training fold phải có cả hai lớp 0 và 1."
        )

    ratio = n_negative / n_positive

    if strategy == "none":
        weight = 1.0

    elif strategy == "balanced":
        weight = ratio

    elif strategy == "sqrt_balanced":
        weight = lib.np.sqrt(ratio)

    else:
        raise ValueError("Chiến lược không hợp lệ!")

    # Giới hạn trọng số tương đối giữa hai lớp
    weight = float(
        lib.np.clip(
            weight,
            1.0 / MAX_WEIGHT,
            MAX_WEIGHT
        )
    )

    return weight


# Minh họa trọng số tại Fold 1
train_idx = fold_data[0]["train_idx"]
y_train = y.iloc[train_idx]

weight_records = []

for label in choice_cols:

    for strategy in WEIGHT_STRATEGIES:

        weight = calculate_class_weight(
            y_train[label],
            strategy
        )

        weight_records.append({
            "Nhãn": label,
            "Strategy": strategy,
            "Positive_Weight": weight
        })

weight_preview = lib.pd.DataFrame(weight_records)

display(
    weight_preview.pivot(
        index="Nhãn",
        columns="Strategy",
        values="Positive_Weight"
    ).round(4)
)

print("PASS: Đã thiết lập các phương pháp gán trọng số.")

In [ ]:
# ==========================================
# MODEL FACTORY - PER-LABEL CLASS WEIGHTS
# ==========================================

# Đọc lại cấu hình V1, giữ nguyên siêu tham số
config_paths = {
    "Random Forest": (
        ROOT_DIR
        / "experiments/v1_baseline/random_forest/results/rf_config.json"
    ),
    "XGBoost": (
        ROOT_DIR
        / "experiments/v1_baseline/xgboost/results/xgb_config.json"
    ),
    "CatBoost": (
        ROOT_DIR
        / "experiments/v1_baseline/catboost/results/cat_config.json"
    )
}

v1_params = {}

for model_name, path in config_paths.items():
    with open(path, "r", encoding="utf-8") as f:
        config = json.load(f)

    v1_params[model_name] = config["hyperparameters"]


def create_weighted_model(model_name, positive_weight):

    params = v1_params[model_name].copy()

    if model_name == "Random Forest":

        params["class_weight"] = {
            0: 1.0,
            1: positive_weight
        }

        return RandomForestClassifier(**params)

    elif model_name == "XGBoost":

        params["scale_pos_weight"] = positive_weight

        return XGBClassifier(**params)

    elif model_name == "CatBoost":

        params["class_weights"] = [
            1.0,
            positive_weight
        ]

        return CatBoostClassifier(**params)

    else:
        raise ValueError("Mô hình không hợp lệ!")


print("PASS: Đã thiết lập trọng số riêng cho từng nhãn.")

In [ ]:
# ==========================================
# TRAIN 3 MODELS WITH CLASS WEIGHTS
# ==========================================

MODEL_NAMES = [
    "Random Forest",
    "XGBoost",
    "CatBoost"
]

def predict_top4(probabilities):

    y_pred = lib.np.zeros_like(
        probabilities,
        dtype=int
    )

    top4_idx = lib.np.argsort(
        -probabilities,
        axis=1,
        kind="stable"
    )[:, :4]

    rows = lib.np.arange(
        len(probabilities)
    )[:, None]

    y_pred[rows, top4_idx] = 1

    return y_pred


experiment_results = []
weight_log = []

for model_name in MODEL_NAMES:

    for strategy in WEIGHT_STRATEGIES:

        print("=" * 60)
        print(f"{model_name} | {strategy}")
        print("=" * 60)

        for fold_info in fold_data:

            fold = fold_info["fold"]
            train_idx = fold_info["train_idx"]
            val_idx = fold_info["val_idx"]

            X_train = X.iloc[train_idx]
            X_val = X.iloc[val_idx]

            y_train = y.iloc[train_idx]
            y_val = y.iloc[val_idx]

            probabilities = []

            for label in choice_cols:

                positive_weight = calculate_class_weight(
                    y_train[label],
                    strategy
                )

                estimator = create_weighted_model(
                    model_name,
                    positive_weight
                )

                # Fit chỉ sử dụng Training
                estimator.fit(
                    X_train,
                    y_train[label]
                )

                classes = list(estimator.classes_)
                positive_index = classes.index(1)

                # Predict chỉ sử dụng Validation
                proba = estimator.predict_proba(
                    X_val
                )[:, positive_index]

                probabilities.append(proba)

                weight_log.append({
                    "Model": model_name,
                    "Strategy": strategy,
                    "Fold": fold,
                    "Nhãn": label,
                    "Positive_Weight": positive_weight
                })

            y_proba = lib.np.column_stack(
                probabilities
            )

            y_pred = predict_top4(y_proba)

            assert y_proba.shape == (len(X_val), 9)
            assert lib.np.isfinite(y_proba).all()
            assert (y_pred.sum(axis=1) == 4).all()

            experiment_results.append({
                "Model": model_name,
                "Strategy": strategy,
                "Fold": fold,
                "val_idx": val_idx,
                "y_true": y_val.to_numpy(dtype=int),
                "y_pred": y_pred,
                "y_proba": y_proba
            })

            print(
                f"Fold {fold}: "
                f"{len(X_train):,} Train | "
                f"{len(X_val):,} Validation | PASS"
            )

weight_log_df = lib.pd.DataFrame(weight_log)

print("\nPASS: Hoàn thành Class Weight Experiment.")
print(f"Số lượt đánh giá: {len(experiment_results)}")

In [ ]:
# ==========================================
# EVALUATION - V2 VS V1
# ==========================================

metric_records = []
label_records = []

for result in experiment_results:

    y_true = result["y_true"]
    y_pred = result["y_pred"]

    info = {
        "Model": result["Model"],
        "Strategy": result["Strategy"],
        "Fold": result["Fold"]
    }

    metric_records.append({
        **info,
        "Exact_Match_Accuracy": accuracy_score(
            y_true, y_pred
        ),
        "Macro_F1": f1_score(
            y_true, y_pred,
            average="macro",
            zero_division=0
        ),
        "Micro_F1": f1_score(
            y_true, y_pred,
            average="micro",
            zero_division=0
        ),
        "Hamming_Loss": hamming_loss(
            y_true, y_pred
        )
    })

    label_scores = f1_score(
        y_true, y_pred,
        average=None,
        zero_division=0
    )

    for label, score in zip(choice_cols, label_scores):
        label_records.append({
            **info,
            "Nhãn": label,
            "F1": score
        })

v2_metrics = lib.pd.DataFrame(metric_records)
v2_label_f1 = lib.pd.DataFrame(label_records)

# Đọc kết quả V1
v1_files = {
    "Random Forest": (
        ROOT_DIR
        / "experiments/v1_baseline/random_forest/results/rf_metrics_5fold.csv"
    ),
    "XGBoost": (
        ROOT_DIR
        / "experiments/v1_baseline/xgboost/results/xgb_metrics_5fold.csv"
    ),
    "CatBoost": (
        ROOT_DIR
        / "experiments/v1_baseline/catboost/results/cat_metrics_5fold.csv"
    )
}

v1_label_files = {
    "Random Forest": (
        ROOT_DIR
        / "experiments/v1_baseline/random_forest/results/rf_label_f1.csv"
    ),
    "XGBoost": (
        ROOT_DIR
        / "experiments/v1_baseline/xgboost/results/xgb_label_f1.csv"
    ),
    "CatBoost": (
        ROOT_DIR
        / "experiments/v1_baseline/catboost/results/cat_label_f1.csv"
    )
}

v1_metric_frames = []
v1_label_frames = []

for model_name in MODEL_NAMES:

    metrics = lib.pd.read_csv(v1_files[model_name])
    labels = lib.pd.read_csv(v1_label_files[model_name])

    metrics["Model"] = model_name
    metrics["Strategy"] = "none"

    labels["Model"] = model_name
    labels["Strategy"] = "none"

    v1_metric_frames.append(metrics)
    v1_label_frames.append(labels)

v1_metrics = lib.pd.concat(
    v1_metric_frames,
    ignore_index=True
)

v1_label_f1 = lib.pd.concat(
    v1_label_frames,
    ignore_index=True
)

# Tổng hợp V1 và V2
all_metrics = lib.pd.concat(
    [v1_metrics, v2_metrics],
    ignore_index=True
)

all_label_f1 = lib.pd.concat(
    [v1_label_f1, v2_label_f1],
    ignore_index=True
)

metric_cols = [
    "Exact_Match_Accuracy",
    "Macro_F1",
    "Micro_F1",
    "Hamming_Loss"
]

comparison = (
    all_metrics
    .groupby(["Model", "Strategy"])[metric_cols]
    .agg(["mean", "std"])
)

print("SO SÁNH V1 VÀ V2 - CLASS WEIGHT")
display(comparison.round(4))

# So sánh F1 từng nhãn
label_comparison = (
    all_label_f1
    .groupby(["Model", "Strategy", "Nhãn"])["F1"]
    .mean()
    .reset_index()
)

for model_name in MODEL_NAMES:

    print(f"\n{model_name} - F1 THEO NHÃN")

    display(
        label_comparison[
            label_comparison["Model"] == model_name
        ].pivot(
            index="Nhãn",
            columns="Strategy",
            values="F1"
        ).round(4)
    )

In [ ]:
# ==========================================
# RARE LABEL IMPACT ANALYSIS
# ==========================================

rare_labels = [
    "Âm nhạc",
    "Mĩ thuật",
    "Sinh học"
]

rare_records = []

for result in experiment_results:

    y_true = result["y_true"]
    y_pred = result["y_pred"]
    y_proba = result["y_proba"]

    for label in rare_labels:

        j = choice_cols.index(label)

        true_label = y_true[:, j]
        pred_label = y_pred[:, j]

        rare_records.append({
            "Model": result["Model"],
            "Strategy": result["Strategy"],
            "Fold": result["Fold"],
            "Nhãn": label,
            "Actual_Positive": int(true_label.sum()),
            "Predicted_Positive": int(pred_label.sum()),
            "True_Positive": int(
                ((true_label == 1) & (pred_label == 1)).sum()
            ),
            "F1": f1_score(
                true_label, pred_label,
                zero_division=0
            ),
            "Mean_Probability": float(
                y_proba[:, j].mean()
            )
        })

rare_impact_df = lib.pd.DataFrame(rare_records)

rare_summary = (
    rare_impact_df
    .groupby(["Model", "Strategy", "Nhãn"])
    .agg({
        "Actual_Positive": "sum",
        "Predicted_Positive": "sum",
        "True_Positive": "sum",
        "F1": "mean",
        "Mean_Probability": "mean"
    })
    .reset_index()
)

# Bổ sung F1 V1 để đối chiếu
v1_rare_f1 = (
    v1_label_f1[
        v1_label_f1["Nhãn"].isin(rare_labels)
    ]
    .groupby(["Model", "Nhãn"])["F1"]
    .mean()
    .reset_index(name="V1_F1")
)

rare_summary = rare_summary.merge(
    v1_rare_f1,
    on=["Model", "Nhãn"],
    how="left",
    validate="many_to_one"
)

rare_summary["Delta_F1_vs_V1"] = (
    rare_summary["F1"] - rare_summary["V1_F1"]
)

print("ẢNH HƯỞNG CỦA CLASS WEIGHT ĐẾN NHÃN HIẾM")
display(rare_summary.round(4))

# Lưu kết quả
reports = {
    "weight_configurations.csv": weight_log_df,
    "v2_metrics_5fold.csv": v2_metrics,
    "v2_label_f1.csv": v2_label_f1,
    "v1_v2_metrics_comparison.csv": (
        comparison.reset_index()
    ),
    "v1_v2_label_f1_comparison.csv": label_comparison,
    "rare_label_impact.csv": rare_impact_df,
    "rare_label_summary.csv": rare_summary
}

for filename, data in reports.items():
    data.to_csv(
        table_dir / filename,
        index=False,
        encoding="utf-8-sig"
    )

# Lưu dự đoán V2 trên Validation
prediction_frames = []

for result in experiment_results:

    val_idx = result["val_idx"]

    pred_df = lib.pd.DataFrame({
        id_col: df_dev.iloc[val_idx][id_col].to_numpy(),
        "Model": result["Model"],
        "Strategy": result["Strategy"],
        "Fold": result["Fold"]
    })

    for j in range(9):
        pred_df[f"true_{j+1}"] = result["y_true"][:, j]
        pred_df[f"pred_{j+1}"] = result["y_pred"][:, j]
        pred_df[f"proba_{j+1}"] = result["y_proba"][:, j]

    prediction_frames.append(pred_df)

v2_predictions = lib.pd.concat(
    prediction_frames,
    ignore_index=True
)

v2_predictions.to_csv(
    table_dir / "v2_class_weight_predictions.csv",
    index=False,
    encoding="utf-8-sig"
)

# Lưu cấu hình thí nghiệm
experiment_config = {
    "experiment": "12 - Class Weight Experiment",
    "models": MODEL_NAMES,
    "strategies": WEIGHT_STRATEGIES,
    "max_weight": MAX_WEIGHT,
    "random_seed": RANDOM_SEED,
    "n_splits": N_SPLITS,
    "cv_source": "data/data-splits/cv_folds_v1",
    "training_source": "Development only",
    "prediction_strategy": "Top-4 probabilities",
    "n_features": 96,
    "n_labels": 9,
    "hyperparameters": v1_params,
    "independent_test_used": False
}

with open(
    config_dir / "class_weight_config.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        experiment_config,
        f,
        ensure_ascii=False,
        indent=4
    )

print("\nPASS: Đã lưu kết quả Class Weight Experiment.")
print("Thư mục:", output_dir.resolve())